#  AgriConnect — Llama-3.2-1B-Instruct Fine-tuning Pipeline
**Model:** `meta-llama/Llama-3.2-1B-Instruct`  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**Dataset:** Fertilizer Prediction CSV + data_core CSV + PDF-extracted JSONL  
**Target:** Crop & fertilizer recommendation from soil test inputs  

**Runtime:** Use Kaggle GPU (T4 x2) or Google Colab Pro (A100)  
Estimated training time: ~1.5–3 hours on T4

##  Step 1 — Install Dependencies

In [1]:
%%capture
%pip install -q \
    transformers==4.46.3 \
    trl==0.12.1 \
    peft==0.13.2 \
    bitsandbytes==0.44.1 \
    accelerate==1.1.1 \
    datasets==3.1.0 \
    scipy

##  Step 2 — Hugging Face Login
Get your token from https://huggingface.co/settings/tokens  
You also need to accept Llama-3.2 license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

In [ ]:
from huggingface_hub import login
login(token="HUGGINGFACE_API_TOKEN")  # Replace with your HF token

##  Step 3 — Upload Your Dataset Files
Upload these 3 files to your Kaggle/Colab session:
- `Fertilizer Prediction.csv`
- `data_core.csv`
- `dataset.jsonl` (from PDF pipeline)

In [4]:
import os
import json
import pandas as pd

# ── Adjust paths if needed ──────────────────────────────────────
FERTILIZER_CSV = "Fertilizer Prediction.csv"
DATA_CORE_CSV  = "data_core.csv"
PDF_JSONL      = "dataset.jsonl"          # from agriconnect_pdf_pipeline.py
OUTPUT_JSONL   = "agriconnect_train.jsonl"
# ────────────────────────────────────────────────────────────────

for f in [FERTILIZER_CSV, DATA_CORE_CSV]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing: {f} — upload it first")
print("Files found ✓")

Files found ✓


##  Step 4 — Build Unified Instruction Dataset

In [5]:
# ── Shared helpers ───────────────────────────────────────────────

SOIL_TYPE_TIPS = {
    "Sandy":  "Sandy soils drain fast; use split fertilizer doses.",
    "Loamy":  "Loamy soils have good nutrient retention.",
    "Black":  "Black (vertisol) soils are rich in calcium and magnesium.",
    "Red":    "Red soils are low in nitrogen and phosphorus.",
    "Clayey": "Clayey soils retain moisture; avoid overwatering.",
}

def soil_tip(soil_type: str) -> str:
    for k, v in SOIL_TYPE_TIPS.items():
        if k.lower() in str(soil_type).lower():
            return v
    return ""

def safe(val, fmt="{}"):
    """Return formatted value or empty string if NaN."""
    try:
        if pd.isna(val):
            return ""
        return fmt.format(val)
    except Exception:
        return str(val)

print("Helpers defined ✓")

Helpers defined ✓


In [6]:
# ── Convert a CSV row → instruction/response pair ────────────────

def row_to_pair(row: pd.Series, source: str) -> dict:
    """
    Works for both CSVs which share identical columns:
    Temparature, Humidity, Moisture, Soil Type, Crop Type,
    Nitrogen, Potassium, Phosphorous, Fertilizer Name
    """
    # Build instruction
    parts = []
    if safe(row.get("Temparature", "")):
        parts.append(f"Temperature: {row['Temparature']}°C")
    if safe(row.get("Humidity", "")):
        parts.append(f"Humidity: {row['Humidity']}%")
    if safe(row.get("Moisture", "")):
        parts.append(f"Moisture: {row['Moisture']}%")
    if safe(row.get("Nitrogen", "")):
        parts.append(f"N: {row['Nitrogen']} kg/ha")
    if safe(row.get("Potassium", "")):
        parts.append(f"K: {row['Potassium']} kg/ha")
    if safe(row.get("Phosphorous", "")):
        parts.append(f"P: {row['Phosphorous']} kg/ha")

    soil  = str(row.get("Soil Type",  "")).strip()
    crop  = str(row.get("Crop Type",  "")).strip()
    fert  = str(row.get("Fertilizer Name", "")).strip()

    if not parts and not soil and not crop:
        return None

    instruction = (
        f"Soil and climate data — {soil} soil, crop: {crop}. "
        f"Readings: {', '.join(parts)}. "
        f"This is for Odisha, India. "
        f"Recommend the best fertilizer with reason."
    )

    tip = soil_tip(soil)
    response = f"Recommended fertilizer: {fert}."
    if tip:
        response += f" Note: {tip}"

    return {
        "instruction": instruction.strip(),
        "response":    response.strip(),
        "source":      source,
    }

print("Converter defined ✓")

Converter defined ✓


In [7]:
all_pairs = []

# ── Fertilizer Prediction CSV ────────────────────────────────────
df1 = pd.read_csv(FERTILIZER_CSV)
df1.columns = [c.strip() for c in df1.columns]
for _, row in df1.iterrows():
    p = row_to_pair(row, "fertilizer_prediction_csv")
    if p: all_pairs.append(p)
print(f"Fertilizer Prediction CSV: {len(df1)} rows -> {sum(1 for p in all_pairs if p['source']=='fertilizer_prediction_csv')} pairs")

# ── data_core CSV ────────────────────────────────────────────────
df2 = pd.read_csv(DATA_CORE_CSV)
df2.columns = [c.strip() for c in df2.columns]
before = len(all_pairs)
for _, row in df2.iterrows():
    p = row_to_pair(row, "data_core_csv")
    if p: all_pairs.append(p)
print(f"data_core CSV:             {len(df2)} rows -> {len(all_pairs)-before} pairs")

# ── PDF-extracted JSONL ──────────────────────────────────────────
pdf_count = 0
if os.path.exists(PDF_JSONL):
    with open(PDF_JSONL, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            obj = json.loads(line)
            if obj.get("instruction") and obj.get("response"):
                all_pairs.append({
                    "instruction": obj["instruction"],
                    "response":    obj["response"],
                    "source":      "pdf_extracted",
                })
                pdf_count += 1
    print(f"PDF JSONL:                 {pdf_count} pairs loaded")
else:
    print(f"PDF JSONL not found — skipping (not required)")

# ── Deduplicate ──────────────────────────────────────────────────
seen = set()
unique_pairs = []
for p in all_pairs:
    key = (p["instruction"][:100], p["response"][:100])
    if key not in seen:
        seen.add(key)
        unique_pairs.append(p)

print(f"\nTotal unique pairs: {len(unique_pairs)}")

Fertilizer Prediction CSV: 100000 rows -> 100000 pairs
data_core CSV:             8000 rows -> 8000 pairs
PDF JSONL:                 132 pairs loaded

Total unique pairs: 76691


In [8]:
# ── Format as Llama-3 chat template & save ───────────────────────

SYSTEM_PROMPT = (
    "You are AgriConnect, an expert agricultural advisor for Odisha, India. "
    "Given soil test readings (N, P, K, temperature, humidity, moisture, soil type, crop type), "
    "you recommend the best fertilizer and dosage schedule. "
    "Be concise, accurate, and practical for Indian farmers."
)

def to_llama3_format(pair: dict) -> dict:
    """
    Llama-3 instruction format:
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system}<|eot_id|>
    <|start_header_id|>user<|end_header_id|>\n{instruction}<|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>\n{response}<|eot_id|>
    """
    text = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{pair['instruction']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n{pair['response']}<|eot_id|>"
    )
    return {"text": text}

formatted = [to_llama3_format(p) for p in unique_pairs]

# Save formatted JSONL
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for item in formatted:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved {len(formatted)} formatted pairs to {OUTPUT_JSONL}")
print(f"\nSample entry:")
print(formatted[0]["text"][:400] + "...")

Saved 76691 formatted pairs to agriconnect_train.jsonl

Sample entry:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are AgriConnect, an expert agricultural advisor for Odisha, India. Given soil test readings (N, P, K, temperature, humidity, moisture, soil type, crop type), you recommend the best fertilizer and dosage schedule. Be concise, accurate, and practical for Indian farmers.<|eot_id|><|start_header_id|>user<|end_header_id|>
Soil and climate ...


##  Step 5 — Load Model + Tokenizer (4-bit QLoRA)

In [1]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model loaded!")
print("Device:", next(model.parameters()).device)

c:\Users\manas\Downloads\Agrionnect\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...
Loading model...


c:\Users\manas\Downloads\Agrionnect\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manas\.cache\huggingface\hub\models--meta-llama--Llama-3.2-1B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model loaded!
Device: cuda:0


##  Step 6 — Apply LoRA Adapters (PEFT)

In [14]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


##  Step 7 — Load Dataset

In [15]:
from datasets import load_dataset
from pathlib import Path

DATA_FILE = Path("agriconnect_train.jsonl")

dataset = load_dataset(
    "json",
    data_files=str(DATA_FILE),
    split="train"
)

dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples : {len(eval_dataset)}")

print("\nColumns:")
print(train_dataset.column_names)

print("\nFirst record:")
print(train_dataset[0])

if "text" in train_dataset.column_names:
    print("\nText preview:")
    print(train_dataset[0]["text"][:300])

Train samples: 69021
Eval samples : 7670

Columns:
['text']

First record:
{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are AgriConnect, an expert agricultural advisor for Odisha, India. Given soil test readings (N, P, K, temperature, humidity, moisture, soil type, crop type), you recommend the best fertilizer and dosage schedule. Be concise, accurate, and practical for Indian farmers.<|eot_id|><|start_header_id|>user<|end_header_id|>\nSoil and climate data — Red soil, crop: Barley. Readings: Temperature: 31°C, Humidity: 69%, Moisture: 42%, N: 27 kg/ha, K: 14 kg/ha, P: 37 kg/ha. This is for Odisha, India. Recommend the best fertilizer with reason.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\nRecommended fertilizer: 20-20. Note: Red soils are low in nitrogen and phosphorus.<|eot_id|>'}

Text preview:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are AgriConnect, an expert agricultural advisor for Odisha, India. Given soil test r

##  Step 8 — Training Arguments

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./agriconnect-llama3-lora",

    # Training
    num_train_epochs=3,
    max_steps=-1,

    # Batch sizes
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Optimizer
    optim="adamw_torch",

    # Learning rate schedule
    learning_rate=2e-4,
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    # Precision
    fp16=True,
    bf16=False,

    # Logging / evaluation
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Reporting
    report_to="none",

    # Stability
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    group_by_length=True,

    # Dataloader
    dataloader_pin_memory=True,
)

print("Training arguments configured successfully ✓")

Training arguments configured successfully ✓


##  Step 9 — Train with SFTTrainer

In [17]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting training...")
trainer.train()
print("Training complete ✓")

Map: 100%|██████████| 7670/7670 [00:00<00:00, 10973.03 examples/s]


Starting training...


                                         
  0%|          | 0/12939 [07:08<?, ?it/s]            

{'loss': 3.2535, 'grad_norm': 21.541290283203125, 'learning_rate': 1.2853470437017995e-05, 'epoch': 0.01}


                                         
  0%|          | 0/12939 [12:44<?, ?it/s]            

...

                                           
  0%|          | 0/12939 [2:31:18<?, ?it/s]             

{'loss': 0.1664, 'grad_norm': 0.6911531686782837, 'learning_rate': 0.000199786642055498, 'epoch': 0.15}


                                           
  0%|          | 0/12939 [2:38:00<?, ?it/s]             

{'loss': 0.2132, 'grad_norm': 0.7741820216178894, 'learning_rate': 0.00019974382965396345, 'epoch': 0.16}


KeyboardInterrupt: 

##  Step 10 — Save Model & Merge Adapters

In [ ]:
# Save LoRA adapter weights only (small — a few MB)
ADAPTER_PATH = "./agriconnect-lora-adapter"
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

In [ ]:
# Merge LoRA weights INTO the base model (for deployment)
from peft import PeftModel

MERGED_PATH = "./agriconnect-merged"

print("Loading base model for merge (in 16-bit)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
merged_model = merged_model.merge_and_unload()   # fuse adapters into weights

merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)

print(f"Merged model saved to {MERGED_PATH} ✓")

##  Step 11 — Inference / Test Your Model

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MERGED_PATH,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

def recommend(temperature, humidity, moisture, soil_type, crop_type, nitrogen, potassium, phosphorous):
    instruction = (
        f"Soil and climate data — {soil_type} soil, crop: {crop_type}. "
        f"Readings: Temperature: {temperature}°C, Humidity: {humidity}%, "
        f"Moisture: {moisture}%, N: {nitrogen} kg/ha, "
        f"K: {potassium} kg/ha, P: {phosphorous} kg/ha. "
        f"This is for Odisha, India. Recommend the best fertilizer with reason."
    )

    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{instruction}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )

    out = pipe(
        prompt,
        max_new_tokens=150,
        do_sample=False,           # greedy decoding = deterministic
        temperature=1.0,
        repetition_penalty=1.1,
    )

    # Strip the prompt prefix from output
    generated = out[0]["generated_text"]
    answer = generated.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    return answer.replace("<|eot_id|>", "").strip()

# ── Test cases ──────────────────────────────────────────────────
print("Test 1 — Sandy soil, Rice, Cuttack-like conditions:")
print(recommend(
    temperature=28, humidity=80, moisture=40,
    soil_type="Sandy", crop_type="Rice",
    nitrogen=37, potassium=0, phosphorous=0
))

print("\nTest 2 — Loamy soil, Wheat, low N:")
print(recommend(
    temperature=22, humidity=65, moisture=35,
    soil_type="Loamy", crop_type="Wheat",
    nitrogen=15, potassium=10, phosphorous=5
))

print("\nTest 3 — Clayey soil, Maize, high moisture:")
print(recommend(
    temperature=30, humidity=85, moisture=60,
    soil_type="Clayey", crop_type="Maize",
    nitrogen=25, potassium=8, phosphorous=12
))

##  Step 12 — Push to Hugging Face Hub (Optional)

In [ ]:
# Uncomment to push to your HF profile
# merged_model.push_to_hub("YOUR_HF_USERNAME/agriconnect-llama3-odisha")
# tokenizer.push_to_hub("YOUR_HF_USERNAME/agriconnect-llama3-odisha")
# print("Model pushed to Hugging Face Hub ✓")

---
## 📋 Quick Reference

| Setting | Value | Notes |
|---|---|---|
| Base model | Llama-3.2-1B-Instruct | ~2.5GB in 4-bit |
| Method | QLoRA (nf4) | ~8GB VRAM needed |
| LoRA rank | 16 | Increase to 32 for more capacity |
| Batch size | 4 × 4 accum = 16 | Reduce if OOM |
| Epochs | 3 | Monitor eval_loss for overfitting |
| Max seq length | 512 tokens | Our prompts are ~100–200 tokens |
| Learning rate | 2e-4 | Standard for LoRA SFT |

**If you get OOM errors:**
- Set `per_device_train_batch_size=2`
- Set `gradient_accumulation_steps=8` (keeps effective batch = 16)
- Set `max_seq_length=256`
- Switch to `fp16=True, bf16=False` on T4 GPU

**Output files:**
- `agriconnect_train.jsonl` — cleaned training data
- `agriconnect-lora-adapter/` — LoRA weights only (for resuming)
- `agriconnect-merged/` — full merged model (for deployment)